# BUSM131 - Telco Customer Churn: Random Survival Forest

Baseline RSF, 5-fold GridSearchCV tuning, concordance index, Integrated Brier Score, permutation importance, customer profile predictions, and risk segmentation. Library: scikit-survival, sklearn.

## 0. INSTALL & IMPORT

In [ ]:
# Uncomment to install:
# !pip install scikit-survival lifelines

%matplotlib inline
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.inspection import permutation_importance

from sksurv.ensemble import RandomSurvivalForest
from sksurv.metrics import brier_score, integrated_brier_score
from sksurv.util import Surv

from lifelines import KaplanMeierFitter

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 7)
plt.rcParams['font.size'] = 12

RANDOM_STATE = 42
print("Libraries loaded successfully.")

## 1. DATA PREPARATION

In [ ]:
print("=" * 65)
print("  1. DATA PREPARATION")
print("=" * 65)

# *** UPDATE THIS PATH to your local file location ***
df = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')

# Fix TotalCharges: blank strings exist for new customers with tenure=0
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'] = df['TotalCharges'].fillna(0)

# --- Survival outcome variables ---
# T = tenure in months (time until churn or censoring)
# E = True if the customer churned (event observed), False if still active (censored)
df['T'] = df['tenure'].clip(lower=0.5)   # avoid zero-duration edge case
df['E'] = (df['Churn'] == 'Yes').astype(bool)

print(f"Total customers : {len(df)}")
print(f"Churned (event) : {df['E'].sum()} ({df['E'].mean()*100:.1f}%)")
print(f"Censored        : {(~df['E']).sum()} ({(~df['E']).mean()*100:.1f}%)")
print(f"Tenure range    : {df['T'].min():.1f} to {df['T'].max():.0f} months")

In [ ]:
# --- Feature engineering: one-hot encode all categorical variables ---
# drop_first=True removes one dummy per group to avoid multicollinearity
cat_cols = ['gender', 'Partner', 'Dependents', 'PhoneService',
            'MultipleLines', 'InternetService', 'OnlineSecurity',
            'OnlineBackup', 'DeviceProtection', 'TechSupport',
            'StreamingTV', 'StreamingMovies', 'Contract',
            'PaperlessBilling', 'PaymentMethod']

df_enc = pd.get_dummies(df, columns=cat_cols, drop_first=True)

# Exclude non-feature columns from model input
# TotalCharges removed: highly collinear with tenure * MonthlyCharges (leakage risk)
drop_cols = ['customerID', 'Churn', 'tenure', 'TotalCharges']
feature_cols = [c for c in df_enc.columns
                if c not in drop_cols + ['T', 'E']]

X = df_enc[feature_cols].astype(float)

# scikit-survival requires a structured numpy array for the outcome
y = Surv.from_arrays(event=df['E'].values, time=df['T'].values)

print(f"Feature matrix : {X.shape[0]} rows x {X.shape[1]} features")
print(f"\nFeatures used:")
for i, f in enumerate(feature_cols):
    print(f"  {i+1:2d}. {f}")

## 2. TRAIN / TEST SPLIT (80/20)

In [ ]:
print("-" * 65)
print("  2. TRAIN / TEST SPLIT (80 / 20, stratified on churn)")
print("-" * 65)

# Stratify ensures both train and test sets preserve the overall 26.5% churn rate
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=df['E'].values
)

print(f"Training set : {X_train.shape[0]} customers ({y_train['event'].mean()*100:.1f}% churn)")
print(f"Test set     : {X_test.shape[0]} customers ({y_test['event'].mean()*100:.1f}% churn)")

## 3. BASELINE RSF MODEL

In [ ]:
print("-" * 65)
print("  3. BASELINE RANDOM SURVIVAL FOREST")
print("-" * 65)

# Baseline RSF: parameters follow standard Random Forest conventions
# (Breiman, 2001; Ishwaran et al., 2008)
# - n_estimators=100: 100 survival trees
# - max_features='sqrt': sqrt(p) features per split (standard RF default)
# - min_samples_leaf=5: controls tree depth / regularisation
rsf_base = RandomSurvivalForest(
    n_estimators=100,
    min_samples_split=10,
    min_samples_leaf=5,
    max_features='sqrt',
    n_jobs=1,
    random_state=RANDOM_STATE
)
rsf_base.fit(X_train, y_train)

c_train_base = rsf_base.score(X_train, y_train)
c_test_base  = rsf_base.score(X_test, y_test)

print(f"\nBaseline RSF:")
print(f"  Concordance Index (train) : {c_train_base:.4f}")
print(f"  Concordance Index (test)  : {c_test_base:.4f}")
print(f"  (1.0 = perfect ordering of churn timing, 0.5 = random chance)")

## 4. HYPERPARAMETER TUNING: 5-FOLD GRIDSEARCHCV

In [ ]:
print("-" * 65)
print("  4. HYPERPARAMETER TUNING - 5-FOLD GRIDSEARCHCV")
print("-" * 65)

# Grid of hyperparameters to search:
#   n_estimators     : more trees = more stable but slower
#   min_samples_leaf : larger = shallower trees = more regularisation
#   max_features     : fewer = more diverse trees; more = lower bias
param_grid = {
    'n_estimators'    : [100, 200],
    'min_samples_leaf': [5, 10, 15],
    'max_features'    : ['sqrt', 0.5],
}

gs = GridSearchCV(
    RandomSurvivalForest(n_jobs=1, random_state=RANDOM_STATE),
    param_grid,
    cv=5,       # 5-fold cross-validation
    n_jobs=1,
    verbose=1,
    refit=True  # refit best model on full training set
)
gs.fit(X_train, y_train)

print(f"\nBest hyperparameters : {gs.best_params_}")
print(f"Best CV C-index      : {gs.best_score_:.4f}")

# Full CV results table
cv_results = (
    pd.DataFrame(gs.cv_results_)
    [['param_n_estimators', 'param_min_samples_leaf',
      'param_max_features', 'mean_test_score',
      'std_test_score', 'rank_test_score']]
    .sort_values('rank_test_score')
    .reset_index(drop=True)
)
print("\nFull CV Results (all 12 combinations):")
print(cv_results.to_string())

## 5. BEST MODEL: EVALUATION ON HELD-OUT TEST SET

In [ ]:
print("-" * 65)
print("  5. BEST MODEL - TEST SET EVALUATION")
print("-" * 65)

rsf_best = gs.best_estimator_

c_train_best = rsf_best.score(X_train, y_train)
c_test_best  = rsf_best.score(X_test,  y_test)

print(f"\nTuned RSF Results:")
print(f"  Concordance Index (train) : {c_train_best:.4f}")
print(f"  Concordance Index (test)  : {c_test_best:.4f}")
print(f"  Overfitting gap           : {c_train_best - c_test_best:.4f}")

# --- Integrated Brier Score (IBS) ---
# Measures calibration of predicted survival probabilities over time.
# IBS = 0 is perfect; IBS = 0.25 = uninformative (flat) model.
# Complements C-index: C-index measures ranking, IBS measures calibration.
times = np.unique(
    np.percentile(df.loc[df['E'], 'T'], np.linspace(10, 90, 15)).astype(int)
)
t_min = y_train['time'].min()
t_max = y_train['time'].max()
times = times[(times >= t_min) & (times <= t_max)]

surv_fns    = rsf_best.predict_survival_function(X_test)
surv_matrix = np.vstack([fn(times) for fn in surv_fns])

try:
    ibs = integrated_brier_score(y_train, y_test, surv_matrix, times)
    print(f"\n  Integrated Brier Score (IBS) : {ibs:.4f}")
    print(f"  (0 = perfect, 0.25 = uninformative)")
except Exception as e:
    print(f"  IBS note: {e}")

## 6. PERMUTATION IMPORTANCE

In [ ]:
print("-" * 65)
print("  6. PERMUTATION IMPORTANCE (evaluated on test set)")
print("-" * 65)

# Permutation importance: randomly shuffles one feature at a time and
# records the resulting drop in concordance index.
# A large drop means the feature is important for ranking churn timing.
# Note: scikit-survival's RSF does not implement feature_importances_
#       (MDI), so permutation importance is the correct approach here.

perm = permutation_importance(
    rsf_best, X_test, y_test,
    n_repeats=10,
    random_state=RANDOM_STATE,
    n_jobs=1
)

perm_df = pd.DataFrame({
    'feature'         : feature_cols,
    'perm_importance' : perm.importances_mean,
    'perm_std'        : perm.importances_std
}).sort_values('perm_importance', ascending=False).reset_index(drop=True)

print("\nTop 15 Features by Permutation Importance:")
print(perm_df.head(15).to_string())
perm_df.to_csv('rsf_permutation_importance.csv', index=False)
top15_perm = perm_df.head(15)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))
bar_colors = ['#e74c3c' if i < 3 else '#f39c12' if i < 7 else '#3498db'
              for i in range(len(top15_perm))]
ax.barh(
    range(len(top15_perm)),
    top15_perm['perm_importance'],
    xerr=top15_perm['perm_std'],
    color=bar_colors,
    edgecolor='black', height=0.6,
    error_kw={'elinewidth': 1.5, 'capsize': 4, 'ecolor': 'grey'}
)
ax.set_yticks(range(len(top15_perm)))
ax.set_yticklabels(top15_perm['feature'], fontsize=10)
ax.set_xlabel('Permutation Importance (drop in Concordance Index)', fontsize=12)
ax.set_title('Random Survival Forest - Permutation Importance (Top 15)',
             fontsize=14, fontweight='bold')
ax.invert_yaxis()
ax.axvline(x=0, color='red', linestyle='--', alpha=0.5, label='No effect')
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

## 7. SURVIVAL PREDICTIONS: CUSTOMER PROFILES

In [ ]:
print("-" * 65)
print("  7. PREDICTED SURVIVAL CURVES - CUSTOMER PROFILES")
print("-" * 65)

# Three archetypal customer profiles based on risk factors identified
# in EDA (Part 1) and Cox PH model (Part 3)
base_profile = X.median().copy()

def make_profile(base, overrides):
    p = base.copy()
    for k, v in overrides.items():
        if k in p.index:
            p[k] = v
    return pd.DataFrame([p])

profiles = {
    'High-Risk:\nMonth-to-month, Fiber optic,\nElectronic check, No security': {
        'Contract_One year': 0, 'Contract_Two year': 0,
        'InternetService_Fiber optic': 1, 'InternetService_No': 0,
        'PaymentMethod_Electronic check': 1,
        'OnlineSecurity_Yes': 0, 'TechSupport_Yes': 0,
        'MonthlyCharges': X['MonthlyCharges'].quantile(0.75)
    },
    'Medium-Risk:\nOne-year contract, DSL,\nAuto-pay, Has security': {
        'Contract_One year': 1, 'Contract_Two year': 0,
        'InternetService_Fiber optic': 0, 'InternetService_No': 0,
        'PaymentMethod_Electronic check': 0,
        'OnlineSecurity_Yes': 1, 'TechSupport_Yes': 1,
        'MonthlyCharges': X['MonthlyCharges'].median()
    },
    'Low-Risk:\nTwo-year contract, DSL,\nAuto-pay, Has security': {
        'Contract_One year': 0, 'Contract_Two year': 1,
        'InternetService_Fiber optic': 0, 'InternetService_No': 0,
        'PaymentMethod_Electronic check': 0,
        'OnlineSecurity_Yes': 1, 'TechSupport_Yes': 1,
        'MonthlyCharges': X['MonthlyCharges'].quantile(0.25)
    }
}

profile_colors = ['#e74c3c', '#f39c12', '#2ecc71']
fig, ax = plt.subplots(figsize=(11, 7))
t_grid = np.arange(1, 73)

print("\nProfile Predictions:")
for (name, overrides), color in zip(profiles.items(), profile_colors):
    p_df  = make_profile(base_profile, overrides)
    sf_fn = rsf_best.predict_survival_function(p_df)[0]
    sf_vals = sf_fn(t_grid)

    below = np.where(sf_vals < 0.5)[0]
    median_t = str(t_grid[below[0]]) if len(below) > 0 else '>72'

    print(f"  {name.replace(chr(10), ' ')}")
    print(f"    Predicted median survival : {median_t} months")
    print(f"    P(survive > 12 months)    : {sf_fn(12):.3f}")
    print(f"    P(survive > 24 months)    : {sf_fn(24):.3f}")
    print()

    ax.plot(t_grid, sf_vals, color=color, lw=2.5, label=name)

ax.axhline(y=0.5, color='grey', linestyle='--', alpha=0.5, label='50% survival')
ax.set_title('RSF Predicted Survival Curves for Customer Profiles',
             fontsize=14, fontweight='bold')
ax.set_xlabel('Tenure (Months)', fontsize=13)
ax.set_ylabel('Survival Probability (Retention)', fontsize=13)
ax.legend(fontsize=9, loc='lower left')
ax.grid(True, alpha=0.3)
ax.set_xlim(0, 72)
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.show()

## 8. CUSTOMER RISK SEGMENTATION (4 TIERS)

In [ ]:
print("-" * 65)
print("  8. CUSTOMER RISK SEGMENTATION")
print("-" * 65)

# Predict cumulative hazard for each customer
# (monotonically related to churn risk: higher = churns sooner)
df['rsf_risk_score'] = rsf_best.predict(X)

# Segment into 4 equal-sized tiers using quartile cut-points
df['rsf_risk_segment'] = pd.qcut(
    df['rsf_risk_score'], q=4,
    labels=['Low Risk', 'Medium Risk', 'High Risk', 'Very High Risk']
)

risk_summary = df.groupby('rsf_risk_segment', observed=True).agg(
    n                   = ('E', 'size'),
    churned             = ('E', 'sum'),
    churn_rate_pct      = ('E', lambda x: x.mean() * 100),
    avg_tenure_months   = ('T', 'mean'),
    avg_monthly_charges = ('MonthlyCharges', 'mean'),
    avg_risk_score      = ('rsf_risk_score', 'mean')
).round(2)

print("\nRisk Segment Summary:")
print(risk_summary.to_string())
risk_summary.to_csv('rsf_risk_segments.csv')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))
risk_colors = {
    'Low Risk'      : '#2ecc71',
    'Medium Risk'   : '#f39c12',
    'High Risk'     : '#e67e22',
    'Very High Risk': '#e74c3c'
}
for segment, color in risk_colors.items():
    mask = df['rsf_risk_segment'] == segment
    kmf_g = KaplanMeierFitter()
    kmf_g.fit(df.loc[mask, 'T'], df.loc[mask, 'E'], label=segment)
    kmf_g.plot_survival_function(ax=ax, ci_show=False, color=color, linewidth=2.5)

ax.set_title('Kaplan-Meier Survival Curves by RSF Risk Segment',
             fontsize=16, fontweight='bold')
ax.set_xlabel('Tenure (Months)', fontsize=13)
ax.set_ylabel('Survival Probability', fontsize=13)
ax.legend(fontsize=12, title='RSF Risk Segment')
ax.grid(True, alpha=0.3)
ax.set_xlim(0, 72)
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
for label, event, color in [('Churned', True, '#e74c3c'), ('Active', False, '#2ecc71')]:
    mask = df['E'] == event
    ax.hist(df.loc[mask, 'rsf_risk_score'], bins=50, alpha=0.6,
            color=color, label=label, edgecolor='black', linewidth=0.5)
ax.set_title('RSF Risk Score Distribution by Actual Churn Outcome',
             fontsize=14, fontweight='bold')
ax.set_xlabel('Cumulative Hazard Score (higher = higher churn risk)')
ax.set_ylabel('Count')
ax.legend(fontsize=12)
plt.tight_layout()
plt.show()

## 9. MODEL COMPARISON: KM vs COX PH vs RSF

In [ ]:
print("=" * 65)
print("  9. FULL MODEL COMPARISON SUMMARY")
print("=" * 65)

comparison = pd.DataFrame({
    'Model'                       : ['Kaplan-Meier (Part 2)', 'Cox PH (Part 3)', 'RSF (Part 4)'],
    'Type'                        : ['Non-parametric', 'Semi-parametric', 'Non-parametric ensemble'],
    'Handles non-linearity'       : ['N/A', 'No', 'Yes'],
    'Handles feature interactions': ['N/A', 'No', 'Yes'],
    'Requires PH assumption'      : ['No', 'Yes', 'No'],
    'C-index (test)'              : ['N/A (descriptive)', '~0.86', f'{c_test_best:.4f}'],
    'Interpretability'            : ['Very High', 'High', 'Moderate (permutation imp.)'],
    'Individual predictions'      : ['No', 'Yes', 'Yes'],
})
print(comparison.to_string(index=False))

print(f"""
KEY RSF FINDINGS:
------------------
Best hyperparameters : {gs.best_params_}
C-index (train)      : {c_train_best:.4f}
C-index (test)       : {c_test_best:.4f}

Top 3 churn predictors (permutation importance):
  1. {perm_df.iloc[0]['feature']}  (delta-C = {perm_df.iloc[0]['perm_importance']:.4f})
  2. {perm_df.iloc[1]['feature']}  (delta-C = {perm_df.iloc[1]['perm_importance']:.4f})
  3. {perm_df.iloc[2]['feature']}  (delta-C = {perm_df.iloc[2]['perm_importance']:.4f})

RSF vs Cox PH:
  - RSF is more flexible (no PH assumption, captures interactions)
  - Cox PH is more interpretable (direct hazard ratios)
  - Both agree on top risk factors: contract type, security services,
    payment method, and fiber optic internet
""")
print("=" * 65)
print("  RANDOM SURVIVAL FOREST ANALYSIS COMPLETE")
print("=" * 65)